# Framework for Project 1

In [1]:
!pip install qiskit

In [2]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
sim = AerSimulator.from_backend(backend)   # noise from real calibration data
lengths = [0, 20, 40, 80, 160, 320]        # all even
shots = 4000

def error_per_gate(gate, qubit=0):
    circs = []
    for n in lengths:
        qc = QuantumCircuit(1, 1)
        for _ in range(n):
            getattr(qc, gate)(0)
            qc.barrier()
        qc.measure(0, 0)
        circs.append(transpile(qc, backend, optimization_level=0,
                               initial_layout=[qubit]))
    result = sim.run(circs, shots=shots).result()
    p0 = np.array([result.get_counts(i).get('0', 0) / shots
                   for i in range(len(lengths))])
    y = 2 * p0 - 1                          # decays as A * d^n
    ok = y > 0
    slope = np.polyfit(np.array(lengths)[ok], np.log(y[ok]), 1)[0]
    return 1 - np.exp(slope)                # error per gate

for q in [0, 1, 2, 3, 4]:                   # repeat on several qubits
    h, x = error_per_gate('h', q), error_per_gate('x', q)
    print(f"qubit {q}: H = {h:.2e}, X = {x:.2e}, H > X? {h > x}")

qubit 0: H = 4.73e-04, X = 3.85e-04, H > X? True
qubit 1: H = 2.77e-04, X = 3.94e-04, H > X? False
qubit 2: H = 6.36e-04, X = 8.01e-04, H > X? False
qubit 3: H = 1.05e-03, X = 1.24e-03, H > X? False
qubit 4: H = 2.48e-04, X = 2.90e-04, H > X? False
